# M and A, workshop 1 — Three files, three dialects

**The situation.** Your company bought a competitor. Their engineer left saying *"at least we had
stored all the information"*, and handed over three customer exports that agree about almost
nothing — separators, column order, capitalisation, even whether there is a header.

Your job is to make them one table in the warehouse schema. Today is step one, the step everyone
skips:

1. **Look at the bytes before you parse.**
2. **Write the target schema down first.**
3. **Load each file into a DataFrame that is actually correct.**

These are the real exports — the same three files you load in your project.

In [ ]:
# ABOUTME: M and A workshop 1 - reading the three real Only Wood customer exports.
# ABOUTME: File dialects, pd.read_csv arguments, and mapping source columns onto a target schema.
import csv
import io
import pandas as pd

from pathlib import Path

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real datasets Qwasar provides.")

TABLE_1 = qwasar_file("only_wood_customer_us_1.csv")
TABLE_2 = qwasar_file("only_wood_customer_us_2.csv")
TABLE_3 = qwasar_file("only_wood_customer_us_3.csv")

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)
print("the three exports Only Wood handed over:")
for p in (TABLE_1, TABLE_2, TABLE_3):
    print(f"  {p.name}: {p.stat().st_size / 1000:.0f} kB, {sum(1 for _ in p.open()):,} lines")

## 1. Look before you parse

`pd.read_csv` will hand you a DataFrame for any of these. A wrong one, usually, and it will not
warn you. So first: print the raw text.

**Rule of thumb:** never call `read_csv` on a file whose first two lines you have not read.

In [ ]:
for path in (TABLE_1, TABLE_2, TABLE_3):
    print(f"----- {path.name}")
    with path.open() as fh:
        for _ in range(2):
            print("   ", fh.readline().rstrip()[:120])
    print()

Three different worlds:

| file | separator | header | notable |
|---|---|---|---|
| **table 1** | `,` | yes | looks normal — it is the trap |
| **table 2** | `;` | **none** | columns in another order, values quoted at random |
| **table 3** | **tab** in the rows, `,` in the header line | sort of | every value carries its type |

Watch what the naive call does to the last two.

In [ ]:
naive_2 = pd.read_csv(TABLE_2)
print("table 2 naive:", naive_2.shape, "<- one column, and the first customer became the header")
display(naive_2.head(2))

naive_3 = pd.read_csv(TABLE_3)
print("table 3 naive:", naive_3.shape, "<- six column names, one column of tab-joined values")
display(naive_3.head(2))

Neither raised an error. Both are wrong: table 2 lost a customer into the header row, and table 3
reports six columns of which five are empty.

**The punchline:** silence is not success. After every load, check `.shape`, `.columns` and
`.head()` against what you saw in the raw text.

## 2. The arguments that fix dialects

| problem | argument |
|---|---|
| separator is not a comma | `sep=";"` or `sep="\t"` |
| there is no header line | `header=None` plus `names=[...]` |
| quotes inside values confuse the parser | `quoting=csv.QUOTE_NONE` |
| the header is not in the same dialect as the rows | read it yourself, then parse the body |

### Exercise 1 — load table 1, and count what is in it

The easy one, to establish the shape of the job.

In [ ]:
table_1 = None      # TODO: a plain read_csv
print(table_1.shape if table_1 is not None else "not yet")
table_1.head(3) if table_1 is not None else None

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("10 000 customers", lambda: len(table_1) == 10000)
check("eight columns", lambda: table_1.shape[1] == 8)
check("it has a UserName column the warehouse does not want", lambda: "UserName" in table_1.columns)

### Exercise 2 — load table 2

No header, `;` separated. From the raw text, the column order is:
`age`, `city`, `gender`, `name`, `email`.

In [ ]:
def load_table_2(path):
    """Return table 2 with columns age, city, gender, name, email."""
    # TODO: sep, header and names - three arguments and you are done
    return pd.DataFrame()

table_2 = load_table_2(TABLE_2)
print(table_2.shape)
table_2.head(3)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("10 000 rows, 5 columns", lambda: table_2.shape == (10000, 5))
check("column names", lambda: list(table_2.columns) == ["age", "city", "gender", "name", "email"])
check("the first customer survived",
      lambda: (table_2.email == "Kutch.lorette@gmail.com").any())

Note the row count: **10 000**, not 9 999. The naive load earlier ate the first customer by
treating them as column names — a whole person silently deleted, and a row count is the only thing
that would have told you.

Two things to notice while you are here, both of which you will have to deal with next week:

- `table_2.isna().sum()` reports **489 rows with no email at all**. Not a placeholder, not a typo:
  simply absent. Email is the only identifier the three systems share, so those 489 customers have
  no key.
- Print `table_2.age.unique()[:20]` and you will find values like `"50years"`. The column is text,
  and not uniformly so.

### Exercise 3 — load table 3

Harder. The first line names the columns, comma separated. Every line after it is **tab**
separated, and the values carry quotes the parser will try to interpret.

Two steps: take the names from line 1, then parse the rest as a tab-separated file.
Hint: `path.read_text().split("\n", 1)` splits off the first line, and `io.StringIO(text)` turns a
string into something `read_csv` can read.

In [ ]:
def load_table_3(path):
    """Return table 3 with the six columns named in its first line."""
    raw = path.read_text()
    # TODO 1: split the first line (the comma separated names) from the body
    # TODO 2: read the body with sep="\t", header=None, names=<those names>,
    #         and quoting=csv.QUOTE_NONE so the stray quotes are left alone
    return pd.DataFrame()

table_3 = load_table_3(TABLE_3)
print(table_3.shape)
table_3.head(3)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("10 000 rows, 6 columns", lambda: table_3.shape == (10000, 6))
check("names taken from line 1",
      lambda: list(table_3.columns) == ["Gender", "Name", "Email", "Age", "City", "Country"])
check("values still carry their type prefix",
      lambda: str(table_3.Gender.iloc[0]).startswith(("string_", "boolean_", "character_")))

Read that last check: table 3 is now **correctly parsed** and still **completely unusable**.
Every value is prefixed with its own type — `string_Male`, `"integer_26"`, `boolean_1`,
`character_M` — as though someone serialised a schema into the data.

Parsing and cleaning are two different jobs. Mixing them is how loaders become unreadable, and
cleaning is next workshop.

## 3. The target schema comes first

You cannot merge three tables into "whatever comes out". You decide the destination, then make
each source arrive there. The warehouse already has a `customers` table:

```
gender, firstname, lastname, email, age, city, country, created_at, referral
```

Nine columns.

### Exercise 4 — the mapping table

For each target column, which source column provides it, per file? `None` where a file does not
have it.

In [ ]:
TARGET_SCHEMA = ["gender", "firstname", "lastname", "email", "age", "city", "country",
                 "created_at", "referral"]

# TODO: complete the three mappings: target column -> source column (or None if absent).
#       Watch out: tables 2 and 3 store the full name in ONE column.
mapping = {
    "table_1": {"gender": "Gender", "firstname": "FirstName"},
    "table_2": {},
    "table_3": {},
}
mapping

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("every file maps every target column",
      lambda: all(set(m) == set(TARGET_SCHEMA) for m in mapping.values()))
check("table 2 has no country", lambda: mapping["table_2"]["country"] is None)
check("one name column feeds two targets",
      lambda: mapping["table_2"]["firstname"] == mapping["table_2"]["lastname"] is not None)
check("nobody has created_at or referral",
      lambda: all(m["created_at"] is None and m["referral"] is None for m in mapping.values()))

### Exercise 5 — what the mapping just told you

Three facts fall out of that table, and each is a decision to make *before* writing merge code.

1. Which target columns does **no** source provide?
2. Table 2 has no country. What goes there?
3. Table 1 has a `UserName` column the schema does not want. What happens to it?

In [ ]:
# TODO: fill in your answers.
missing_everywhere = []        # target columns no file provides
table_2_country_plan = ""      # one short sentence
username_plan = ""             # one short sentence

print(missing_everywhere, "|", table_2_country_plan, "|", username_plan)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("the two orphan columns", lambda: sorted(missing_everywhere) == ["created_at", "referral"])
check("both decisions written down",
      lambda: len(table_2_country_plan) > 20 and len(username_plan) > 20)

There is no single right answer to 2 and 3 — there is a *defensible* one. In a migration, the
decisions you can explain six months later are worth more than the clever ones.

## 4. All three, side by side

In [ ]:
loaded = {"table 1": table_1, "table 2": table_2, "table 3": table_3}
if any(df is None or df.empty for df in loaded.values()):
    print("Finish exercises 1 to 3, then run this cell for the summary.")
else:
    for name, df in loaded.items():
        print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols | {', '.join(df.columns)}")
    print(f"\n{sum(len(d) for d in loaded.values()):,} rows in total, before any de-duplication")

Three correct DataFrames, 30 000 rows. Not three *clean* ones — look at them and you will still
find `string_Male`, a country called `12`, `INDIANAPOLIS` next to `new_york`, an email address
shared by hundreds of customers, and a gender column that speaks six different languages.

That is next week.

## What to take away

1. **Read the raw text first.** Two minutes of looking saves an afternoon of wrong numbers.
2. **A load that does not raise is not a load that worked.** Check shape, columns, and one row —
   the naive table-2 load silently deleted a customer.
3. **`read_csv` has an argument for every dialect** — `sep`, `header`, `names`, `quoting`. You are
   not supposed to memorise them, you are supposed to reach for them.
4. **Write the target schema before the merge code.** It turns a vague "combine these" into a
   checklist of per-column decisions.
5. **Parse first, clean second.** Separate functions, separate workshop.

### Take it home
Write your three loaders as functions and assert the shape of each result. If `my_m_and_a` starts
by proving all three files loaded correctly, everything after it is a pleasure instead of a hunt.